In [13]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# =========================================================
# Configuration
# =========================================================

RESULTS_DIR = Path("results")
BASELINE_DIR = RESULTS_DIR / "baselines"
PLOT_DIR = RESULTS_DIR / "plots"

SA_RESULTS_DIR = Path("..") / "testSA"

SA_DIRS = {
    "cell":            SA_RESULTS_DIR / "cell_graph_res",
    "graph":           SA_RESULTS_DIR / "cell_graph_res",
    "cell_inverted":   SA_RESULTS_DIR / "cell_graph_res_inverted",
    "graph_inverted":  SA_RESULTS_DIR / "cell_graph_res_inverted",
    "vehicle":         SA_RESULTS_DIR / "veic_res",
    "vehicle_inverted":SA_RESULTS_DIR / "veic_res_inverted",
}

SCENARIOS = {
    "cell":    {"trace": "c_and_g"},
    "graph":   {"trace": "c_and_g"},
    "vehicle": {"trace": "vehicle"},
}

# ---------------------------------------------------------
# Style
# ---------------------------------------------------------

SA_COLOR     = "tab:blue"
DENSE_COLOR  = "tab:orange"
RANDOM_COLOR = "tab:green"

Y_LIM   = (0.0, 1.05)
X_TICKS = [0, 400, 800, 1200]
Y_TICKS      = [0.0, 0.25, 0.50, 0.75, 1.0]
Y_TICKLABELS = ["0", ".25", ".50", ".75", "1"]

# =========================================================
# SA raw reading
# =========================================================

def read_sa_raw(experiment_dir: Path, scenario: str) -> pd.DataFrame:
    rows = []
    for rsu_dir in experiment_dir.glob("rsu_*"):
        if not rsu_dir.is_dir():
            continue
        try:
            rsus_from_dir = int(rsu_dir.name.removeprefix("rsu_"))
        except ValueError:
            continue

        raw_file = rsu_dir / "tuning_raw.csv"
        if not raw_file.is_file():
            continue

        df = pd.read_csv(raw_file, sep=";")
        df = df[df["scenario"].str.lower() == scenario.lower()].copy()
        if df.empty:
            continue

        if not (df["rsus"] == rsus_from_dir).all():
            raise ValueError(f"RSU mismatch in {raw_file}")

        rows.append(df)

    if not rows:
        raise RuntimeError(f"No tuning_raw.csv data for {scenario} in {experiment_dir}")

    return pd.concat(rows, ignore_index=True)


def aggregate_sa(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("rsus")
        .agg(
            c0_mean=("best_c0_s", "mean"),
            c0_std=("best_c0_s", "std"),
            c1_mean=("best_c1_s", "mean"),
            c1_std=("best_c1_s", "std"),
            c2_mean=("best_c2_s", "mean"),
            c2_std=("best_c2_s", "std"),
            runs=("seed", "count"),
        )
        .reset_index()
        .sort_values("rsus")
    )

# =========================================================
# Worst-group utility
# =========================================================

def worst_group_from_sa(sa_agg: pd.DataFrame) -> pd.DataFrame:
    out = sa_agg.copy()
    out["worst_mean"] = out[["c0_mean", "c1_mean", "c2_mean"]].min(axis=1)
    out["worst_std"]  = out[["c0_std",  "c1_std",  "c2_std"]].min(axis=1)
    return out


def worst_group_from_dense(dense_df: pd.DataFrame) -> pd.DataFrame:
    out = dense_df.copy()
    out["worst"] = out[["c0_s", "c1_s", "c2_s"]].min(axis=1)
    return out


def worst_group_from_random(random_df: pd.DataFrame) -> pd.DataFrame:
    per_run = random_df.copy()
    per_run["worst"] = per_run[["c0_s", "c1_s", "c2_s"]].min(axis=1)
    return (
        per_run.groupby("rsus")
        .agg(
            worst_mean=("worst", "mean"),
            worst_std=("worst", "std"),
        )
        .reset_index()
        .sort_values("rsus")
    )

# =========================================================
# Plotting
# =========================================================

def plot_objective_comparison(name: str,
                              trace: str,
                              sa_agg: pd.DataFrame,
                              dense_df: pd.DataFrame,
                              random_df: pd.DataFrame) -> None:

    plt.rcParams.update({
        # IEEE / Times-like fonts
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Nimbus Roman", "Liberation Serif", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",

        # Sizes
        "font.size": 20,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "legend.fontsize": 20,
    })

    sa_w    = worst_group_from_sa(sa_agg).sort_values("rsus")
    dense_w = worst_group_from_dense(dense_df).sort_values("rsus")
    rand_w  = worst_group_from_random(random_df).sort_values("rsus")

    common = sorted(
        set(sa_w["rsus"])
        .intersection(dense_w["rsus"])
        .intersection(rand_w["rsus"])
    )
    if not common:
        raise ValueError(f"{name}: no common RSU budgets across methods.")

    sa_w    = sa_w[sa_w["rsus"].isin(common)]
    dense_w = dense_w[dense_w["rsus"].isin(common)]
    rand_w  = rand_w[rand_w["rsus"].isin(common)]

    # X axis is just the RSU budget itself.
    x = np.array(common, dtype=float)

    fig, ax = plt.subplots(figsize=(6.5, 5))

    # SA — blue, solid, circles, ± std band
    ax.plot(x, sa_w["worst_mean"], color=SA_COLOR,
            marker="o", linestyle="-", label="SA")
    if sa_w["worst_std"].notna().any():
        ax.fill_between(
            x,
            sa_w["worst_mean"] - sa_w["worst_std"].fillna(0),
            sa_w["worst_mean"] + sa_w["worst_std"].fillna(0),
            color=SA_COLOR, alpha=0.15, linewidth=0,
        )

    # Dwell-rank — orange, dashed, squares, deterministic
    ax.plot(x, dense_w["worst"], color=DENSE_COLOR,
            marker="s", linestyle="--", label="Dwell-rank")

    # Random — green, dotted, triangles, ± std band
    ax.plot(x, rand_w["worst_mean"], color=RANDOM_COLOR,
            marker="^", linestyle=":", label="Random")
    if rand_w["worst_std"].notna().any():
        ax.fill_between(
            x,
            rand_w["worst_mean"] - rand_w["worst_std"].fillna(0),
            rand_w["worst_mean"] + rand_w["worst_std"].fillna(0),
            color=RANDOM_COLOR, alpha=0.15, linewidth=0,
        )

    ax.set_xlabel(r"RSU budget $\it{B}$")
    ax.set_ylabel("Worst-group utility")

    ax.set_xticks(X_TICKS)
    ax.set_yticks(Y_TICKS)
    ax.set_yticklabels(Y_TICKLABELS)
    ax.set_ylim(*Y_LIM)
    ax.set_xlim(left=0)

    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="lower right", frameon=True)

    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{name}_objective.png", dpi=600)
    fig.savefig(PLOT_DIR / f"{name}_objective.pdf", dpi=600)
    plt.close(fig)

# =========================================================
# Main
# =========================================================

def main() -> None:
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    for name in SA_DIRS.keys():
        base_name   = name.removesuffix("_inverted")
        sa_scenario = base_name
        trace       = SCENARIOS[base_name]["trace"]

        raw    = read_sa_raw(SA_DIRS[name], sa_scenario)
        sa_agg = aggregate_sa(raw)

        dense_path  = BASELINE_DIR / f"{name}_dense.csv"
        random_path = BASELINE_DIR / f"{name}_random.csv"

        if not dense_path.is_file() or not random_path.is_file():
            print(f"[skip] {name}: missing {dense_path.name} or {random_path.name}")
            continue

        dense_df  = pd.read_csv(dense_path,  sep=";")
        random_df = pd.read_csv(random_path, sep=";")

        plot_objective_comparison(name, trace, sa_agg, dense_df, random_df)
        print(f"[ok] {name}: wrote {PLOT_DIR / f'{name}_objective.png'}")


if __name__ == "__main__":
    main()

[ok] cell: wrote results/plots/cell_objective.png
[ok] graph: wrote results/plots/graph_objective.png
[ok] cell_inverted: wrote results/plots/cell_inverted_objective.png
[ok] graph_inverted: wrote results/plots/graph_inverted_objective.png
[ok] vehicle: wrote results/plots/vehicle_objective.png
[ok] vehicle_inverted: wrote results/plots/vehicle_inverted_objective.png
